In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

# <a target="_blank" href="https://colab.research.google.com/github/facebookresearch/sam3/blob/main/notebooks/sam3_image_batched_inference.ipynb">
#   <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
# </a>

In [ ]:
using_colab = False

In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    python_exe = sys.executable
    !{python_exe} -m pip install opencv-python matplotlib scikit-learn huggingface_hub
    !{python_exe} -m pip install 'git+https://github.com/facebookresearch/sam3.git'

In [ ]:
from pathlib import Path
import os

def _load_export_style_env_file(env_path: Path):
    loaded_keys = []
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value
        loaded_keys.append(key)
    return loaded_keys

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / ".env").exists():
    repo_root = repo_root.parent

env_file = repo_root / ".env"
if env_file.exists():
    loaded_env_keys = _load_export_style_env_file(env_file)
    print(f"Loaded {len(loaded_env_keys)} vars from {env_file}")
else:
    loaded_env_keys = []
    print("No .env file found while walking up from current working directory")

hf_home = os.environ.get("HF_HOME", "").strip()
if hf_home:
    os.makedirs(hf_home, exist_ok=True)
    if "HF_HUB_CACHE" not in os.environ:
        os.environ["HF_HUB_CACHE"] = hf_home if hf_home.endswith("/hub") else f"{hf_home}/hub"

hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    try:
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False)
        print("Hugging Face authentication configured from .env")
    except Exception as exc:
        print(f"Hugging Face login skipped: {exc}")

from PIL import Image
import requests
from io import BytesIO
import sam3

sam3_root = os.path.abspath(os.path.join(os.path.dirname(sam3.__file__), ".."))
print(f"SAM3 root: {sam3_root}")
print(f"HF_HOME: {os.environ.get('HF_HOME', '<default>')}")

In [ ]:
import torch
# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    # use bfloat16 for the entire notebook. If your card doesn't support it, try float16 instead
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    compute_device = torch.device("cuda")
else:
    compute_device = torch.device("cpu")

# inference mode for the whole notebook. Disable if you need gradients
torch.inference_mode().__enter__()
print(f"Compute device: {compute_device}")

# Utilities

## Plotting

This section contains simple utilities to plot masks and bounding masks on top of an image

In [ ]:
# Plot utility from sam3.visualization_utils is not needed for prompt-free mode.
# We render masks with matplotlib in the plotting section below.

## Batching

This section contains some utility functions to create datapoints. They are optional, but give some good indication on how they should be created

In [ ]:
from sam3.train.data.sam3_image_dataset import InferenceMetadata, FindQueryLoaded, Image as SAMImage, Datapoint
from typing import List

GLOBAL_COUNTER = 1
def create_empty_datapoint():
    """ A datapoint is a single image on which we can apply several queries at once. """
    return Datapoint(find_queries=[], images=[])

def set_image(datapoint, pil_image):
    """ Add the image to be processed to the datapoint """
    w,h = pil_image.size
    datapoint.images = [SAMImage(data=pil_image, objects=[], size=[h,w])]

def add_text_prompt(datapoint, text_query):
    """ Add a text query to the datapoint """

    global GLOBAL_COUNTER
    # in this function, we require that the image is already set.
    # that's because we'll get its size to figure out what dimension to resize masks and boxes
    # In practice you're free to set any size you want, just edit the rest of the function
    assert len(datapoint.images) == 1, "please set the image first"

    w, h = datapoint.images[0].size
    datapoint.find_queries.append(
        FindQueryLoaded(
            query_text=text_query,
            image_id=0,
            object_ids_output=[], # unused for inference
            is_exhaustive=True, # unused for inference
            query_processing_order=0,
            inference_metadata=InferenceMetadata(
                coco_image_id=GLOBAL_COUNTER,
                original_image_id=GLOBAL_COUNTER,
                original_category_id=1,
                original_size=[w, h],
                object_id=0,
                frame_index=0,
            )
        )
    )
    GLOBAL_COUNTER += 1
    return GLOBAL_COUNTER - 1

def add_visual_prompt(datapoint, boxes:List[List[float]], labels:List[bool], text_prompt="visual"):
    """ Add a visual query to the datapoint.
    The bboxes are expected in XYXY format (top left and bottom right corners)
    For each bbox, we expect a label (true or false). The model tries to find boxes that ressemble the positive ones while avoiding the negative ones
    We can also give a text_prompt as an additional hint. It's not mandatory, leave it to "visual" if you want the model to solely rely on the boxes.

    Note that the model expects the prompt to be consistent. If the text reads "elephant" but the provided boxe points to a dog, the results will be undefined.
    """

    global GLOBAL_COUNTER
    # in this function, we require that the image is already set.
    # that's because we'll get its size to figure out what dimension to resize masks and boxes
    # In practice you're free to set any size you want, just edit the rest of the function
    assert len(datapoint.images) == 1, "please set the image first"
    assert len(boxes) > 0, "please provide at least one box"
    assert len(boxes) == len(labels), f"Expecting one label per box. Found {len(boxes)} boxes but {len(labels)} labels"
    for b in boxes:
        assert len(b) == 4, f"Boxes must have 4 coordinates, found {len(b)}"

    labels = torch.tensor(labels, dtype=torch.bool).view(-1)
    if not labels.any().item() and text_prompt=="visual":
        print("Warning: you provided no positive box, nor any text prompt. The prompt is ambiguous and the results will be undefined")
    w, h = datapoint.images[0].size
    datapoint.find_queries.append(
        FindQueryLoaded(
            query_text=text_prompt,
            image_id=0,
            object_ids_output=[], # unused for inference
            is_exhaustive=True, # unused for inference
            query_processing_order=0,
            input_bbox=torch.tensor(boxes, dtype=torch.float).view(-1,4),
            input_bbox_label=labels,
            inference_metadata=InferenceMetadata(
                coco_image_id=GLOBAL_COUNTER,
                original_image_id=GLOBAL_COUNTER,
                original_category_id=1,
                original_size=[w, h],
                object_id=0,
                frame_index=0,
            )
        )
    )
    GLOBAL_COUNTER += 1
    return GLOBAL_COUNTER - 1

# Loading

First we load our model

In [ ]:
from pathlib import Path
from sam3 import build_sam3_image_model

sam3_pkg_dir = Path(sam3.__file__).resolve().parent
sam3_root = str(sam3_pkg_dir.parent)

bpe_candidates = [
    Path(sam3_root) / "assets" / "bpe_simple_vocab_16e6.txt.gz",
    sam3_pkg_dir / "assets" / "bpe_simple_vocab_16e6.txt.gz",
]
bpe_path = next((str(p) for p in bpe_candidates if p.exists()), None)
if bpe_path is None:
    raise FileNotFoundError(
        f"Could not find SAM3 BPE vocab file. Checked: {[str(p) for p in bpe_candidates]}"
    )

model = build_sam3_image_model(bpe_path=bpe_path, enable_inst_interactivity=True)
model = model.to(compute_device)
model.eval()
print(f"SAM3 model loaded on {compute_device}")

Then our validation transforms

In [ ]:
from sam3.train.transforms.basic_for_api import ComposeAPI, RandomResizeAPI, ToTensorAPI, NormalizeAPI

transform = ComposeAPI(
    transforms=[
        RandomResizeAPI(sizes=1008, max_size=1008, square=True, consistent_transform=False),
        ToTensorAPI(),
        NormalizeAPI(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ]
)


And finally our postprocessor

In [ ]:
from sam3.eval.postprocessors import PostProcessImage
postprocessor = PostProcessImage(
    max_dets_per_img=-1,       # if this number is positive, the processor will return topk. For this demo we instead limit by confidence, see below
    iou_type="segm",           # we want masks
    use_original_sizes_box=True,   # our boxes should be resized to the image size
    use_original_sizes_mask=True,   # our masks should be resized to the image size
    convert_mask_to_rle=False, # the postprocessor supports efficient conversion to RLE format. In this demo we prefer the binary format for easy plotting
    detection_threshold=0.5,   # Only return confident detections
    to_cpu=False,
)

# Inference

For inference, we proceed as follows:
- Create each datapoint one by one, using the functions above. Each query that we make will give us a unique id, which is then used after post-processing to retrieve the results
- Each datapoint must be transformed according to are pre-processing transforms (basically resize to 1008x1008, normalize)
- We then collate all datapoints into a batch and forward it to the model

In [ ]:
# Prompt-free setup: load images only (no text or visual prompts)
img1 = Image.open("/mnt/abka03/xlvlm_data/imagenet_3_class/train/cat/n02123045_9839.JPEG").convert("RGB")
img2 = Image.open("/mnt/abka03/xlvlm_data/imagenet_3_class/train/rabbit/n02325366_6190.JPEG").convert("RGB")
print(f"Loaded image 1 size: {img1.size}")
print(f"Loaded image 2 size: {img2.size}")

In [ ]:
# Optional local image example
img_local_example = None
# img_local_example = Image.open(f"{sam3_root}/assets/images/test_image.jpg").convert("RGB")

images_for_auto = [img1, img2]
if img_local_example is not None:
    images_for_auto.append(img_local_example)

print(f"Running prompt-free segmentation on {len(images_for_auto)} image(s)")

In [ ]:
# Prompt-free SAM3 segmentation (point-grid, no text prompt)
import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.sam3_utils import predict_auto_masks_sam3

sam3_model_dict = {
    "model": model,
    "confidence_threshold": 0.7,
}

auto_pairs_per_img = predict_auto_masks_sam3(
    sam3_model_dict,
    images_for_auto,
    topn=32,
    min_mask_area=100,
    points_per_side=24,
)

auto_pairs_img1 = auto_pairs_per_img[0]
auto_pairs_img2 = auto_pairs_per_img[1]
print(f"Image 1 masks: {len(auto_pairs_img1)}")
print(f"Image 2 masks: {len(auto_pairs_img2)}")

In [ ]:
# Quick summary of largest masks by area
def _top_mask_areas(pairs, k=10):
    areas = [int(mask.sum()) for _, mask in pairs if mask is not None]
    areas.sort(reverse=True)
    return areas[:k]

print("Image 1 top mask areas:", _top_mask_areas(auto_pairs_img1))
print("Image 2 top mask areas:", _top_mask_areas(auto_pairs_img2))

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def plot_auto_masks_grid(image, pairs, max_masks=12, cols=4, alpha=0.5, seed=42):
    if len(pairs) == 0:
        print("No masks found")
        return

    n = min(max_masks, len(pairs))
    rows = math.ceil(n / cols)
    rng = np.random.default_rng(seed)

    _, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(rows, cols)

    for idx in range(rows * cols):
        ax = axes[idx // cols, idx % cols]
        ax.axis("off")
        if idx >= n:
            continue

        bbox, mask = pairs[idx]
        overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
        color = rng.random(3)
        overlay[..., :3] = color
        overlay[..., 3] = mask.astype(np.float32) * alpha

        ax.imshow(image)
        ax.imshow(overlay)
        x, y, w, h = bbox
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="yellow", linewidth=1.5))
        ax.set_title(f"#{idx+1} area={int(mask.sum())}")

    plt.tight_layout()
    plt.show()

# Plotting

In [ ]:
# Image 1: top automatic masks
plot_auto_masks_grid(img1, auto_pairs_img1, max_masks=5, cols=4, alpha=0.5, seed=1)

In [ ]:
# Image 2: top automatic masks
plot_auto_masks_grid(img2, auto_pairs_img2, max_masks=5, cols=4, alpha=0.5, seed=2)

In [ ]:
# Image 1: more masks
plot_auto_masks_grid(img1, auto_pairs_img1, max_masks=24, cols=6, alpha=0.45, seed=3)

In [ ]:
# Image 2: more masks
plot_auto_masks_grid(img2, auto_pairs_img2, max_masks=24, cols=6, alpha=0.45, seed=4)

In [ ]:
# Bounding boxes summary for image 1 (top 10)
for idx, (bbox, mask) in enumerate(auto_pairs_img1[:10], start=1):
    print(f"img1 mask {idx:02d}: bbox={bbox}, area={int(mask.sum())}")

In [ ]:
# Bounding boxes summary for image 2 (top 10)
for idx, (bbox, mask) in enumerate(auto_pairs_img2[:10], start=1):
    print(f"img2 mask {idx:02d}: bbox={bbox}, area={int(mask.sum())}")

In [ ]:
# Final prompt-free segmentation summary
print({
    "image_1_num_masks": len(auto_pairs_img1),
    "image_2_num_masks": len(auto_pairs_img2),
    "points_per_side": 24,
    "topn": 32,
    "min_mask_area": 100,
})